# Entrenamiento reproducible en Google Colab (Kaggle como alternativa)
Ejecute las celdas en orden en un entorno **GPU CUDA**. El camino principal es **Google Colab** (L4, A100 o T4): el código sale del ZIP de Drive y los datos, modelos y reportes se conservan en Drive. **Kaggle** es la alternativa cuando Colab no asigna GPU (**GPU T4 x2**, no P100): allí se trabaja en `/tmp` y solo el ZIP de resultados queda en `/kaggle/working` (pestaña *Output*). Tras cada etapa v2 se actualiza el ZIP `quant-ai-resultados_<sesión>.zip` (en Drive o en `/kaggle/working`) y, antes de exportar, una celda resume qué etapas terminaron bien y cuáles fallaron. No ejecute este notebook en el PC de trabajo. TPU no está soportado. El holdout final permanece excluido. `SMOKE=True` sirve únicamente para comprobar el flujo; sus resultados no constituyen evidencia de rendimiento.

## 0. Parámetros
Colab (camino principal): suba el ZIP del código a `Mi unidad/quant-ai/quant-ai.zip` o use un repositorio accesible en `REPO_URL`. Kaggle (alternativa): suba `quant-ai-kaggle.zip` como dataset privado y agréguelo con *Add Input*; la sección 3 lo encuentra sola. La plataforma se detecta sola por las variables de entorno; si la primera celda imprime un entorno equivocado, fije `PLATAFORMA = "colab"` o `"kaggle"`. Mantenga `RUN_TAG` y los hiperparámetros para reanudar; cámbielo para iniciar otro experimento. Copie las unidades/hora que muestra su sesión de Colab; no se asume ninguna tarifa.

In [ ]:
REPO_URL = ""  # Opcional; deje vacío para usar el ZIP de Drive.
CODE_ZIP = "/content/drive/MyDrive/quant-ai/quant-ai.zip"
DRIVE_DIR = "/content/drive/MyDrive/quant-ai"
SMOKE = False
PLATAFORMA = "auto"  # "auto", "colab" o "kaggle". Cámbielo solo si la detección automática se equivoca.
MODELOS = ["gru", "tcn"]
UNITS_PER_HOUR = None  # Copiar de Colab; no es un precio monetario.
RUN_TAG = "secuencias_v1"
SEED = 42
EPOCHS = 30
BATCH_SIZE = 512
LAST_FOLDS = 4
NUM_WORKERS = 2
COMPILE = False
# Etapas a ejecutar. Las de v1 ya corrieron el 2026-11-09 (resultados en Drive): repetirlas da lo mismo.
# "pruebas" corre en la nube las pruebas que en el PC se omiten por falta de torch; si fallan, la corrida sigue.
ETAPAS = {"pruebas": True, "ml_v1": False, "benchmark": False, "secuencias_v1": False, "rl": False,
          "datos_externos": True, "experimento_v2": True, "secuencias_v2": True, "cnn": True}

import os, sys, time, json, subprocess, shutil, importlib.util, uuid
from pathlib import Path
from datetime import datetime, timezone
SESSION_STARTED = time.perf_counter()
SESSION_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
KAGGLE_INPUT = "/kaggle/input"            # Kaggle: datasets agregados con "Add Input" (solo lectura)
KAGGLE_HOME = "/tmp/qa_home"   # Kaggle: datos y modelos en /tmp; a /kaggle/working (la salida) va solo el ZIP final
# Ni la carpeta /kaggle/input ni el paquete google.colab distinguen la plataforma: Colab trae esa carpeta
# (kagglehub) y la imagen de Kaggle puede traer ese paquete. Se usan las variables de entorno de cada una.
try:
    _colab_pkg = importlib.util.find_spec("google.colab") is not None
except (ModuleNotFoundError, ValueError):
    _colab_pkg = False
_colab_env = any(k.startswith("COLAB_") for k in os.environ)
# KAGGLE_KERNEL_RUN_TYPE solo existe dentro de un notebook de Kaggle; KAGGLE_URL_BASE cuenta si no hay señales de Colab.
_kaggle_env = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or (bool(os.environ.get("KAGGLE_URL_BASE")) and not _colab_env)
if PLATAFORMA not in ("auto", "colab", "kaggle"):
    raise ValueError('PLATAFORMA debe ser "auto", "colab" o "kaggle".')
if PLATAFORMA == "auto":
    IN_KAGGLE = _kaggle_env
    IN_COLAB = (not IN_KAGGLE) and (_colab_env or _colab_pkg)
else:
    IN_KAGGLE, IN_COLAB = PLATAFORMA == "kaggle", PLATAFORMA == "colab"
IN_CLOUD = IN_COLAB or IN_KAGGLE
if IN_COLAB:
    WORK_DIR, SCRATCH = Path("/content/quant-ai"), Path("/content")
elif IN_KAGGLE:
    WORK_DIR, SCRATCH = Path("/tmp/quant-ai"), Path("/tmp")   # código fuera de la salida de Kaggle
else:
    WORK_DIR, SCRATCH = Path.cwd(), Path.cwd()
print("Entorno:", "Colab" if IN_COLAB else "Kaggle" if IN_KAGGLE else "local (no se entrena aquí)",
      f"| detección: PLATAFORMA={PLATAFORMA}, variables Kaggle={_kaggle_env}, variables Colab={_colab_env}, "
      f"paquete google.colab={_colab_pkg}")
if not MODELOS or any(m not in {"gru", "tcn", "transformer"} for m in MODELOS):
    raise ValueError("MODELOS debe contener gru, tcn o transformer.")


## 1. Detectar el entorno
Se consulta el hardware asignado, sin suponer un modelo de GPU ni resultados de rendimiento.

In [ ]:
import platform
print("Python:", sys.version)
print("CPU lógicas:", os.cpu_count())
print("Plataforma:", platform.platform())
if Path("/proc/meminfo").exists():
    print("\n".join(Path("/proc/meminfo").read_text().splitlines()[:3]))
disk = shutil.disk_usage(str(SCRATCH))
print({k: round(v / 2**30, 2) for k, v in zip(["disco_total_GB", "disco_usado_GB", "disco_libre_GB"], disk)})
import torch
print("torch:", torch.__version__, "CUDA:", torch.version.cuda, "cuDNN:", torch.backends.cudnn.version())
if torch.cuda.is_available():
    prop = torch.cuda.get_device_properties(0)
    print("GPU:", prop.name, "VRAM GiB:", prop.total_memory / 2**30)
    print("Compute capability:", torch.cuda.get_device_capability(0))
    print("bf16:", torch.cuda.is_bf16_supported(), "| GPUs visibles:", torch.cuda.device_count(), "(se usa la 0)")
    if torch.cuda.get_device_capability(0) < (7, 0):
        print("AVISO: esta GPU (p. ej. P100) no es compatible con el PyTorch instalado. En Kaggle elija GPU T4 x2.")
else:
    print("Sin CUDA. Active una GPU antes del entrenamiento completo "
          "(Colab: Cambiar tipo de entorno; Kaggle: Settings > Accelerator > GPU T4 x2).")
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)


## 2. Montar Drive (Colab) o usar `/tmp/qa_home` (Kaggle) y fijar las rutas persistentes
El modo continúa siendo PAPER. Las instalaciones y el entrenamiento de este notebook se bloquean fuera de Colab y Kaggle. En Colab `QUANT_AI_HOME` es la carpeta de Drive (`DRIVE_DIR`): datos, modelos y reportes sobreviven a una desconexión. En Kaggle viven en `/tmp/qa_home` y se pierden al cerrar la sesión; a `/kaggle/working` (la salida) solo va el ZIP de resultados, que se actualiza tras cada etapa v2. En Kaggle no hay reanudación entre sesiones.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    HOME_DIR = DRIVE_DIR
elif IN_KAGGLE:
    HOME_DIR = KAGGLE_HOME
else:
    HOME_DIR = None
    print("Fuera de Colab y Kaggle: no se monta nada ni se instala nada.")
if HOME_DIR:
    Path(HOME_DIR).mkdir(parents=True, exist_ok=True)
    os.environ["QUANT_AI_HOME"] = HOME_DIR
    os.environ["QUANT_AI_MODE"] = "paper"
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    print("QUANT_AI_HOME:", HOME_DIR, "| modo:", os.environ["QUANT_AI_MODE"])


## 3. Obtener el código e instalar únicamente lo necesario
El ZIP puede contener los archivos directamente o una carpeta superior. En Kaggle el código se busca en `/kaggle/input` (ZIP o carpeta ya descomprimida por Kaggle) y se copia a `/tmp`, porque `/kaggle/input` es de solo lectura. Si el paquete trae `estado_previo/` (datos limpios e historial de intentos para el Deflated Sharpe), se copia a `QUANT_AI_HOME` sin sobrescribir nada. Con el ZIP, una copia anterior del código en la VM se reemplaza por la del ZIP más reciente (datos, modelos y reportes viven en `QUANT_AI_HOME`, no ahí); con `REPO_URL` se conserva el checkout existente. No se reinstala torch. Aquí también se definen `run_step` (si un script falla, repite sus últimas 40 líneas bajo «ÚLTIMAS LÍNEAS DEL ERROR» para copiarlas en el chat), la lista de fallos y `export_results`. El proyecto requiere Python 3.10–3.12; si el runtime no cumple, seleccione uno compatible antes de continuar.

In [ ]:
if not IN_CLOUD:
    raise RuntimeError("Esta celda solo se ejecuta en Colab o Kaggle; no instala paquetes en el PC.")
if IN_KAGGLE:
    # En Kaggle Internet viene apagado por defecto. Sin él, la corrida gastaría horas de GPU sin datos externos.
    import urllib.request
    try:
        urllib.request.urlopen("https://data.binance.vision/", timeout=15).close()
    except Exception as error:  # noqa: BLE001
        if getattr(error, "code", None) is None:  # sin respuesta HTTP: no hay red
            raise RuntimeError("Sin Internet: actívalo en Settings > Internet del notebook de Kaggle "
                               f"y vuelve a ejecutar ({type(error).__name__}).") from error
    print("Internet: disponible.")
if sys.version_info[:2] < (3, 10):
    raise RuntimeError("quant-ai requiere Python >= 3.10.")
if sys.version_info[:2] >= (3, 13):
    print(f"Aviso: Python {sys.version.split()[0]} no se ha probado con quant-ai (probado en 3.11); se continúa.")
import zipfile, collections
# Usar siempre el ZIP más reciente (Drive en Colab, /kaggle/input en Kaggle): si el entorno sobrevivió de otra
# sesión, el código viejo se reemplaza (los datos, modelos y reportes viven en QUANT_AI_HOME, no aquí).
if WORK_DIR.exists() and not REPO_URL:
    os.chdir(SCRATCH)
    shutil.rmtree(WORK_DIR)


def find_kaggle_code(root):
    """Busca el proyecto en los datasets agregados. Kaggle suele descomprimir el ZIP: se acepta carpeta o ZIP.

    Recorre con os.walk siguiendo enlaces (Kaggle puede montar cada dataset como enlace simbólico, y
    Path.rglob no los sigue). Si no lo encuentra, el error muestra lo que hay en /kaggle/input.
    """
    projects, zips, seen = [], [], []
    for folder, dirs, files in os.walk(root, followlinks=True):
        depth = len(Path(folder).relative_to(root).parts)
        if depth <= 2:
            seen.append(f"  {folder}: carpetas {sorted(dirs)[:8]}, archivos {sorted(files)[:8]}")
        if "pyproject.toml" in files and (Path(folder) / "src" / "quant_ai").is_dir():
            projects.append(Path(folder))
            dirs[:] = []            # no buscar dentro del propio proyecto
            continue
        zips += [Path(folder) / f for f in files if f.startswith("quant-ai") and f.endswith(".zip")]
        if depth >= 6:
            dirs[:] = []
    projects, zips = sorted(projects), sorted(zips)
    if len(projects) == 1:
        return projects[0]
    if not projects and len(zips) == 1:
        return zips[0]
    if not projects and not zips:
        reason = ("Este notebook no tiene agregado el dataset con el código. En el panel derecho: Add Input > "
                  "Your Datasets > elige el dataset donde subiste quant-ai.zip (o quant-ai-kaggle.zip) y "
                  "vuelve a ejecutar esta celda.")
    else:
        reason = ("Hay más de una copia del código. Deja un solo dataset de quant-ai en Input (quita el viejo) "
                  f"y vuelve a ejecutar. Proyectos: {projects}; ZIP: {zips}")
    raise FileNotFoundError(f"No encontré un único proyecto en {root}. {reason}\nContenido de {root}:\n"
                            + ("\n".join(seen[:25]) if len(seen) > 1 or (seen and "[]" not in seen[0])
                               else "  (vacío: no hay ningún dataset agregado)"))


if not (WORK_DIR / "pyproject.toml").exists():
    if REPO_URL:
        subprocess.run(["git", "clone", REPO_URL, str(WORK_DIR)], check=True)
    else:
        archive_path = find_kaggle_code(Path(KAGGLE_INPUT)) if IN_KAGGLE else Path(CODE_ZIP)
        if not (archive_path.is_file() or archive_path.is_dir()):
            raise FileNotFoundError(f"Suba el ZIP a {archive_path}")
        print("Código desde:", archive_path)
    if not REPO_URL and archive_path.is_dir():
        shutil.copytree(archive_path, WORK_DIR, dirs_exist_ok=True)
    elif not REPO_URL:
        staging = SCRATCH / ("quant-ai-unpack-" + uuid.uuid4().hex[:8])
        staging.mkdir()
        with zipfile.ZipFile(archive_path) as archive:
            for entry in archive.infolist():
                target = (staging / entry.filename).resolve()
                if not target.is_relative_to(staging.resolve()):
                    raise ValueError("El ZIP contiene una ruta fuera de la carpeta de extracción.")
            archive.extractall(staging)
        candidates = list(staging.rglob("pyproject.toml"))
        if len(candidates) != 1:
            raise ValueError("El ZIP debe contener exactamente un proyecto, sin .venv.")
        shutil.copytree(candidates[0].parent, WORK_DIR, dirs_exist_ok=True)
os.chdir(WORK_DIR)
STATE_DIR = WORK_DIR / "estado_previo"
if STATE_DIR.is_dir() and os.environ.get("QUANT_AI_HOME"):
    home_dir, restored = Path(os.environ["QUANT_AI_HOME"]), 0
    for f in sorted(STATE_DIR.rglob("*")):
        dest = home_dir / f.relative_to(STATE_DIR)
        if f.is_file() and not dest.exists():
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, dest)
            restored += 1
    print(f"estado_previo: {restored} archivos copiados a {home_dir} (los existentes no se tocan).")
required_files = ["scripts/train_sequence.py", "scripts/gpu_benchmark.py", "configs/base.yaml"]
for filename in required_files:
    if not Path(filename).is_file():
        raise FileNotFoundError(f"El código está incompleto: falta {filename}. Incluya los archivos nuevos en el ZIP.")
editable = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], check=False)
if editable.returncode != 0:
    print("Aviso: pip install -e falló (¿sin Internet?). Se usa PYTHONPATH=src, que basta para este notebook.")
packages = {"pyarrow": "pyarrow", "yaml": "pyyaml", "lightgbm": "lightgbm", "sklearn": "scikit-learn"}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)
# Un kernel ya iniciado no lee el .pth del modo editable hasta reiniciarse:
# se agrega src/ a la ruta de este kernel y a PYTHONPATH para los scripts lanzados con subprocess.
SRC_DIR = str(WORK_DIR / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
os.environ["PYTHONPATH"] = os.pathsep.join(filter(None, [SRC_DIR, os.environ.get("PYTHONPATH")]))
importlib.invalidate_caches()
import pandas as pd
from quant_ai.config import load_config
CFG = load_config(WORK_DIR / "configs/base.yaml")
TIMING_FILE = CFG.reports_dir / "colab_sessions" / f"{SESSION_ID}.json"
TIMING_FILE.parent.mkdir(parents=True, exist_ok=True)
STEPS = []
FALLOS = []  # etapas que fallaron sin detener la corrida; la celda de resumen las muestra al final


def run_step(label, command, log_path=None):
    """Ejecuta un script mostrando su salida en vivo (stdout y stderr unidos).

    Si falla, repite sus últimas 40 líneas para copiarlas en el chat y las guarda en STEPS; con log_path
    además guarda la salida completa en ese archivo.
    """
    started = time.perf_counter()
    status, tail = "failed", collections.deque(maxlen=40)
    log = open(log_path, "w", encoding="utf-8") if log_path else None
    try:
        # PYTHONUNBUFFERED: sin él, la salida de los scripts llegaría en bloques y no en vivo.
        with subprocess.Popen(command, cwd=WORK_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, encoding="utf-8", errors="replace",
                              env={**os.environ, "PYTHONUNBUFFERED": "1"}) as proc:
            try:
                for line in proc.stdout:
                    print(line, end="", flush=True)
                    tail.append(line.rstrip("\n"))
                    if log:
                        log.write(line)
                proc.wait()
            except BaseException:
                proc.kill()  # igual que subprocess.run: una interrupción no deja el script corriendo en la GPU
                proc.wait()
                raise
        if proc.returncode:
            raise subprocess.CalledProcessError(proc.returncode, command, output="\n".join(tail))
        status = "finished"
    except KeyboardInterrupt:
        status = "interrupted"
        raise
    except Exception:
        print("\n" + "=" * 78 + f"\nÚLTIMAS LÍNEAS DEL ERROR en «{label}» (cópialas en el chat):\n"
              + "\n".join(tail) + "\n" + "=" * 78, flush=True)
        raise
    finally:
        if log:
            log.close()
        step = {"step": label, "seconds": time.perf_counter() - started, "status": status}
        if status != "finished":
            step["tail"] = list(tail)
        STEPS.append(step)
        TIMING_FILE.write_text(json.dumps({"session": SESSION_ID, "steps": STEPS,
            "units_per_hour": UNITS_PER_HOUR, "elapsed_seconds": time.perf_counter() - SESSION_STARTED},
            indent=2, ensure_ascii=False), encoding="utf-8")


def registrar_fallo(etapa, error):
    """Anota una etapa caída sin detener la corrida (con las últimas líneas de su salida, si las hay)."""
    tail = getattr(error, "output", None)
    FALLOS.append({"etapa": etapa, "tipo": type(error).__name__, "mensaje": str(error),
                   "ultimas_lineas": tail.splitlines() if isinstance(tail, str) else []})
    print(f"ETAPA {etapa} FALLÓ: {type(error).__name__}: {error}")
    print("Se continúa con las demás etapas; pega este error en el chat para corregirlo.")


# Colab: el ZIP queda en Drive (DRIVE_DIR). Kaggle: en /kaggle/working, la única carpeta que se guarda como salida.
EXPORT_ZIP = (Path("/kaggle/working") if IN_KAGGLE else Path(HOME_DIR)) / f"quant-ai-resultados_{SESSION_ID}.zip"


def export_results(final=False):
    """Empaqueta reportes, métricas, predicciones y registro de modelos (sin checkpoints .pt/.ckpt) en EXPORT_ZIP.

    El ZIP se arma en el disco local de la VM y se copia de una vez al destino con reemplazo atómico: Drive
    recibe una sola escritura secuencial y nunca queda un ZIP a medias. La exportación intermedia (tras cada
    etapa v2) no detiene la corrida si falla; la final sí lanza el error.
    """
    started = time.perf_counter()
    home = Path(os.environ["QUANT_AI_HOME"])
    local = SCRATCH / f"{EXPORT_ZIP.stem}.{uuid.uuid4().hex[:8]}.tmp"
    staged = EXPORT_ZIP.with_name(EXPORT_ZIP.name + ".tmp")
    try:
        n = 0
        with zipfile.ZipFile(local, "w", zipfile.ZIP_DEFLATED) as z:
            for rel in ["reports", "models/dl", "models/rl", "models/registry"]:
                base = home / rel
                for f in sorted(base.rglob("*")) if base.exists() else []:
                    if f.is_file() and f.suffix not in {".pt", ".ckpt", ".zip", ".tmp"} and "checkpoints" not in f.parts:
                        z.write(f, f.relative_to(home).as_posix())
                        n += 1
        shutil.copyfile(local, staged)
        os.replace(staged, EXPORT_ZIP)
        print(f"{'Exportación final' if final else 'ZIP intermedio actualizado'}: {n} archivos -> {EXPORT_ZIP} "
              f"({EXPORT_ZIP.stat().st_size / 1e6:.1f} MB, {time.perf_counter() - started:.1f} s)")
        return EXPORT_ZIP
    except Exception as error:  # noqa: BLE001
        if final:
            raise
        registrar_fallo("exportación intermedia", error)
    finally:
        for temporary in (local, staged):
            temporary.unlink(missing_ok=True)

CONFIG_ARGS = ["--config", str(WORK_DIR / "configs/base.yaml")]
# Guardar versiones y huella del código para reproducir el experimento.
import hashlib, importlib.metadata
versions = {p: importlib.metadata.version(p) for p in ["torch", "numpy", "pandas", "pyarrow", "scikit-learn", "lightgbm"]}
source_paths = sorted([*Path("src").rglob("*.py"), *Path("scripts").glob("*.py"), Path("configs/base.yaml")])
hashes = {str(p): hashlib.sha256(p.read_bytes()).hexdigest() for p in source_paths}
(CFG.reports_dir / f"environment_{SESSION_ID}.json").write_text(
    json.dumps({"python": sys.version, "versions": versions, "source_sha256": hashes,
                "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version(),
                "seed": SEED}, indent=2), encoding="utf-8")
print("Proyecto:", WORK_DIR, "Datos:", CFG.data_dir)


## 3b. Pruebas en la nube
Ejecuta con torch las pruebas que en el PC se omiten (secuencias, CNN, indicadores técnicos, experimento v2 y backtest), antes de entrenar. Si algo falla, la corrida sigue: el fallo queda en el resumen final y la salida completa en `reports/colab_sessions/pytest_<sesión>.txt`, que viaja en el ZIP. `tests/test_rl_env.py` solo se agrega si la etapa `rl` está activa (y se omite sola si gymnasium aún no está instalado).

In [ ]:
try:
    if ETAPAS["pruebas"]:
        if IN_CLOUD and importlib.util.find_spec("pytest") is None:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=True)
        test_files = ["tests/test_sequence.py", "tests/test_chart_cnn.py", "tests/test_technical.py",
                      "tests/test_experiment_v2.py", "tests/test_backtest.py"]
        if ETAPAS["rl"]:
            test_files.append("tests/test_rl_env.py")
        # -q ya viene en addopts de pyproject.toml (repetirlo ocultaría la línea de totales); -rs muestra por qué
        # se omite una prueba: en la nube ninguna de torch debería omitirse.
        run_step("pruebas", [sys.executable, "-m", "pytest", *test_files, "-p", "no:cacheprovider", "-rs"],
                 log_path=CFG.reports_dir / "colab_sessions" / f"pytest_{SESSION_ID}.txt")
    else:
        print("Etapa pruebas omitida.")

except Exception as error:  # noqa: BLE001 — las pruebas no deben impedir el entrenamiento
    registrar_fallo("pruebas", error)


## 4. Descargar o actualizar los datos
Este paso consulta las fuentes del proyecto y conserva los datos y su manifiesto en `QUANT_AI_HOME` (Drive en Colab). La descarga inicial también se realiza en modo smoke. En Kaggle, si `estado_previo/` trajo los datos limpios del PC y cubren todo el periodo de desarrollo, no se descargan de nuevo: el experimento usa exactamente los mismos datos que las corridas anteriores. En Colab y en Kaggle, si la descarga falla (p. ej. la API REST de Binance bloqueada en la región de la VM), se reintenta solo con los archivos mensuales (`--no-rest`). Si también falla pero los datos guardados llegan hasta la última vela antes del holdout, se continúa con un aviso; si no, la celda se detiene y dice qué hacer. Ante cualquier fallo se imprimen las últimas líneas del error para copiarlas en el chat.

In [ ]:
clean_files = [CFG.data_dir / "clean" / f"{s}_{CFG.interval}.parquet" for s in CFG.symbols]


def datos_cubren_desarrollo():
    """True si hay Parquet limpio de cada símbolo y llega a la última vela anterior al holdout final."""
    last_needed = CFG.final_holdout_start - pd.Timedelta(CFG.interval)
    try:
        return all(p.is_file() and pd.to_datetime(pd.read_parquet(p, columns=["open_time"])["open_time"],
                                                    utc=True).max() >= last_needed for p in clean_files)
    except Exception as error:  # noqa: BLE001 — un Parquet ilegible equivale a no tener datos
        print(f"No se pudieron leer los datos limpios: {type(error).__name__}: {error}")
        return False


if IN_KAGGLE and datos_cubren_desarrollo():
    print("Datos limpios de estado_previo/ presentes: no se descargan de nuevo (mismos datos del PC).")
    STEPS.append({"step": "datos", "seconds": 0.0, "status": "skipped_bundled"})
else:
    try:
        run_step("datos", [sys.executable, "scripts/download_data.py", *CONFIG_ARGS])
    except subprocess.CalledProcessError:
        print("La descarga falló (ver las últimas líneas arriba); se reintenta solo con los archivos mensuales "
              "de data.binance.vision (--no-rest).")
        try:
            run_step("datos_sin_rest", [sys.executable, "scripts/download_data.py", *CONFIG_ARGS, "--no-rest"])
        except subprocess.CalledProcessError as error:
            if not datos_cubren_desarrollo():
                raise RuntimeError(
                    "La descarga falló dos veces y no hay datos limpios que cubran el periodo de desarrollo. "
                    "Copia en el chat las ÚLTIMAS LÍNEAS DEL ERROR que aparecen arriba. Para seguir sin descargar, "
                    f"sube BTCUSDT_1h.parquet y ETHUSDT_1h.parquet de data/clean del PC a {CFG.data_dir / 'clean'} "
                    "y vuelve a ejecutar esta celda.") from error
            registrar_fallo("datos", error)
            print(f"AVISO: no se pudieron actualizar los datos, pero los guardados en {CFG.data_dir / 'clean'} "
                  "cubren todo el periodo de desarrollo (hasta el holdout). Se continúa con ellos.")


## 5. Experimento ML en CPU
La ejecución normal realiza el experimento completo y registra el candidato con las reglas existentes. En smoke solo usa un fold y un símbolo, sin registro. El entrenamiento secuencial no cambia el modelo activo.

In [ ]:
if ETAPAS["ml_v1"]:
    ml_command = [sys.executable, "scripts/run_experiments.py", *CONFIG_ARGS, "--n-jobs", "-1"]
    if SMOKE:
        ml_command += ["--max-folds", "1", "--symbols", "BTCUSDT", "--tag", "smoke_colab"]
    else:
        ml_command += ["--register"]
    run_step("ml_cpu", ml_command)

else:
    print("Etapa ml_v1 omitida (ver ETAPAS en la celda de parámetros).")


## 5b. Datos de otras plataformas
Derivados de Binance (funding, prima, open interest, ratios long/short), prima de Coinbase, volatilidad implícita de Deribit (DVOL), Fear & Greed, stablecoins, macro (FRED) y amplitud de altcoins y on-chain. Cada dato lleva su hora real de disponibilidad (`available_at`). Los originales van a disco local (`/content/qa_cache` en Colab, `/tmp/qa_cache` en Kaggle); los Parquet consolidados quedan en Drive (Colab) o en `/tmp/qa_home` (Kaggle). Si una fuente está bloqueada en la región de la VM (p. ej. Deribit en EE. UU.), se avisa y sus columnas quedan vacías.


In [ ]:
try:
    if ETAPAS["datos_externos"]:
        CACHE_DIR = ("/content/qa_cache" if IN_COLAB else "/tmp/qa_cache" if IN_KAGGLE
                     else str(CFG.data_dir / "external"))
        ext_cmd = [sys.executable, "scripts/download_extended.py", *CONFIG_ARGS, "--workers", "8",
                   "--cache-dir", CACHE_DIR]
        if SMOKE:
            ext_cmd += ["--since", "2025-09-01", "--symbols", "BTCUSDT"]
        run_step("datos_externos", ext_cmd)
    else:
        print("Etapa datos_externos omitida.")

except Exception as error:  # noqa: BLE001 — una etapa caída no debe impedir consolidar ni exportar
    registrar_fallo("datos_externos", error)
if ETAPAS["datos_externos"]:
    export_results()  # el ZIP de resultados se actualiza tras cada etapa v2 (un corte no borra lo terminado)


## 5c. Experimento v2: ablación, meta-etiquetado y volatility targeting (CPU)
Variantes pre-registradas en `configs/experiments_v2.yaml`, todas con el mismo walk-forward (2022-01 a 2025-10) y tres escenarios de costes. El reporte incluye el Deflated Sharpe Ratio, que corrige por TODOS los intentos acumulados, y la puerta de promoción. El holdout final sigue sellado.


In [ ]:
try:
    if ETAPAS["experimento_v2"]:
        v2_cmd = [sys.executable, "scripts/run_experiments_v2.py", *CONFIG_ARGS, "--n-jobs", "-1"]
        if SMOKE:
            v2_cmd += ["--max-folds", "1", "--symbols", "BTCUSDT", "--tag", "v2_smoke_colab"]
        run_step("experimento_v2", v2_cmd)
        latest_v2 = sorted((CFG.reports_dir / "runs").glob("*_v2*"))[-1]
        print((latest_v2 / "report.md").read_text(encoding="utf-8"))
    else:
        print("Etapa experimento_v2 omitida.")

except Exception as error:  # noqa: BLE001 — una etapa caída no debe impedir consolidar ni exportar
    registrar_fallo("experimento_v2", error)
if ETAPAS["experimento_v2"]:
    export_results()  # el ZIP de resultados se actualiza tras cada etapa v2 (un corte no borra lo terminado)


## 6. Benchmark GPU
Las filas y el número de características provienen del dataset de desarrollo real. La duración por época es una estimación de ajuste, sin validación, E/S ni checkpoints. El modo smoke del benchmark usa una prueba pequeña en CPU.

In [ ]:
if ETAPAS["benchmark"]:
    from quant_ai.training.dataset import build_datasets
    benchmark_data = build_datasets(CFG, final=False)
    rows = [f"{symbol}={len(ds.features)}" for symbol, ds in benchmark_data.items()]
    n_features = len(next(iter(benchmark_data.values())).feature_names)
    del benchmark_data
    benchmark_command = [sys.executable, "scripts/gpu_benchmark.py", *CONFIG_ARGS,
                         "--n-features", str(n_features), "--rows-per-symbol", *rows]
    if SMOKE:
        benchmark_command.append("--smoke")
    run_step("benchmark", benchmark_command)
    import pandas as pd
    latest_benchmark = max((CFG.reports_dir / "gpu_benchmark").glob("*.json"), key=lambda p: p.stat().st_mtime)
    benchmark = json.loads(latest_benchmark.read_text())
    display(pd.DataFrame(benchmark["matmul"]))
    display(pd.json_normalize(benchmark["training"]))

else:
    print("Etapa benchmark omitida (ver ETAPAS en la celda de parámetros).")


## 7. Entrenar modelos secuenciales y reanudar
La carpeta es estable entre sesiones. Reejecute esta celda con los mismos parámetros: `--resume` comienza desde cero si no hay checkpoint y retoma la última época completa si existe. Los folds terminados no vuelven a entrenarse. Smoke usa una carpeta distinta, un símbolo, un fold, dos épocas y ventana 16.

In [ ]:
if ETAPAS["secuencias_v1"]:
    if not torch.cuda.is_available() and not SMOKE:
        raise RuntimeError("Active una GPU CUDA antes de entrenar; TPU no está soportado.")
    RUN_ROOT = CFG.models_dir / "dl" / (RUN_TAG + ("_smoke" if SMOKE else ""))
    RUN_DIRS = {}
    for model_name in MODELOS:
        output = RUN_ROOT / model_name
        RUN_DIRS[model_name] = output
        command = [sys.executable, "scripts/train_sequence.py", *CONFIG_ARGS,
            "--model", model_name, "--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE),
            "--last-folds", str(LAST_FOLDS), "--seed", str(SEED), "--num-workers", str(NUM_WORKERS),
            "--out", str(output), "--resume"]
        # TCN: 4 bloques dilatados (1, 2, 4, 8) => campo receptivo de 61 velas, casi toda la ventana de 64.
        command += {"tcn": ["--layers", "4"]}.get(model_name, [])
        if SMOKE:
            command.append("--smoke")
        if COMPILE:
            command.append("--compile")
        run_step(f"secuencia_{model_name}", command)

else:
    print("Etapa secuencias_v1 omitida (ver ETAPAS en la celda de parámetros).")


## 7b. GRU/TCN v2: todas las características (base + técnico + externo), horizonte 24 h
Mismo tramo de prueba que el experimento v2 (16 trimestres, 2022-2025) para compararlos de igual a igual. Con `--separate-validation` el early stopping usa su propia validación (último 20 % del ajuste, purgada) y la calibración queda solo para Platt, como en la CNN. En SMOKE se borra antes la carpeta `secuencias_v2_smoke/<modelo>`: es desechable y un manifiesto viejo impediría `--resume`.


In [ ]:
try:
    if ETAPAS["secuencias_v2"]:
        if not torch.cuda.is_available() and not SMOKE:
            raise RuntimeError("Active una GPU CUDA antes de entrenar.")
        RUN_DIRS_V2 = {}
        for model_name in MODELOS:
            out = CFG.models_dir / "dl" / ("secuencias_v2" + ("_smoke" if SMOKE else "")) / model_name
            RUN_DIRS_V2[model_name] = out
            if SMOKE and out.exists():
                shutil.rmtree(out)  # prueba mínima desechable: un manifiesto viejo impediría --resume
            seq_cmd = [sys.executable, "scripts/train_sequence.py", *CONFIG_ARGS, "--model", model_name,
                       "--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE), "--last-folds", "16",
                       "--seed", str(SEED), "--num-workers", str(NUM_WORKERS), "--out", str(out), "--resume",
                       "--feature-groups", "base,ta,ext", "--horizon", "24", "--test-start", "2022-01-01",
                       "--separate-validation"]
            seq_cmd += {"tcn": ["--layers", "4"]}.get(model_name, [])
            if SMOKE:
                seq_cmd.append("--smoke")
            run_step(f"secuencia_v2_{model_name}", seq_cmd)
            res = json.loads((out / "backtest_summary.json").read_text(encoding="utf-8"))
            display(pd.DataFrame(res["strategies"]).T[["total_return", "sharpe", "max_drawdown", "n_trades"]])
    else:
        print("Etapa secuencias_v2 omitida.")

except Exception as error:  # noqa: BLE001 — una etapa caída no debe impedir consolidar ni exportar
    registrar_fallo("secuencias_v2", error)
if ETAPAS["secuencias_v2"]:
    export_results()  # el ZIP de resultados se actualiza tras cada etapa v2 (un corte no borra lo terminado)


## 8. Evaluación y comparación
El CLI ya evalúa al terminar. Esta celda permite regenerar la evaluación desde las predicciones. Cada tabla muestra su periodo y compara las estrategias sobre los mismos símbolos y precios. No mezcle tablas de periodos distintos con las métricas del experimento ML completo.

In [ ]:
if ETAPAS["secuencias_v1"]:
    comparisons = []
    for model_name, output in RUN_DIRS.items():
        run_step(f"evaluacion_{model_name}", [sys.executable, "scripts/evaluate_sequence.py",
            *CONFIG_ARGS, "--pred", str(output / "predictions.parquet"), "--name", f"dl_{model_name}"])
        summary = json.loads((output / "backtest_summary.json").read_text())
        table = pd.DataFrame.from_dict(summary["strategies"], orient="index")
        table.insert(0, "experimento", model_name)
        table.insert(1, "inicio", summary["period"]["start"])
        table.insert(2, "fin", summary["period"]["end"])
        comparisons.append(table)
    display(pd.concat(comparisons).rename_axis("estrategia"))

else:
    print("Etapa secuencias_v1 omitida (ver ETAPAS en la celda de parámetros).")


## 9. Presupuesto de la sesión
Se mide el tiempo transcurrido desde la celda 0, incluidos los tiempos de espera entre celdas. La conversión usa únicamente la tasa ingresada por usted. Es una estimación de unidades para esta sesión, no una factura. Si cambia de GPU o de tarifa, comience otra sesión de registro. Los cortes abruptos pueden dejar tiempo sin registrar.

In [ ]:
elapsed = time.perf_counter() - SESSION_STARTED
budget = {"session": SESSION_ID, "elapsed_seconds": elapsed, "elapsed_hours": elapsed / 3600,
          "units_per_hour": UNITS_PER_HOUR, "estimated_units": None, "steps": STEPS}
if UNITS_PER_HOUR is not None:
    rate = float(UNITS_PER_HOUR)
    if rate < 0:
        raise ValueError("UNITS_PER_HOUR no puede ser negativo.")
    budget["estimated_units"] = elapsed / 3600 * rate
else:
    print("No se calculan unidades: UNITS_PER_HOUR no fue ingresado.")
TIMING_FILE.write_text(json.dumps(budget, indent=2), encoding="utf-8")
display(pd.DataFrame([{k: v for k, v in budget.items() if k != "steps"}]))
display(pd.DataFrame(STEPS))


## Aprendizaje por refuerzo
PPO o DQN, un activo por modelo, tres semillas y resultados en `QUANT_AI_HOME` (Drive en Colab). Entrena hasta el cierre de 2023 y evalúa 2024-01-01 a 2025-10-31; el script rechaza el holdout final. Con MLP, SB3 suele ir igual o mejor en CPU: mida el mismo smoke en CPU y CUDA con nombres de ejecución distintos y deje que ese benchmark decida (el benchmark secuencial no mide RL). [Referencia de SB3](https://stable-baselines3.readthedocs.io/en/master/modules/ppo.html). Para ejecutar solo RL, use las celdas 0–4 y esta sección.

In [ ]:
if ETAPAS["rl"]:
    if not IN_CLOUD:
        raise RuntimeError("RL solo se instala y entrena en Colab o Kaggle; no ejecutar localmente.")
    rl_packages = {"gymnasium": "gymnasium", "stable_baselines3": "stable-baselines3"}
    rl_missing = [package for module, package in rl_packages.items()
                  if importlib.util.find_spec(module) is None]
    if rl_missing:
        subprocess.run([sys.executable, "-m", "pip", "install", *rl_missing], check=True)
    RL_ALGO = "ppo"  # También admite dqn.
    RL_DEVICE = "cpu"  # Cambie a cuda solo para medirlo en el mismo problema.
    RL_RUN_TAG = "rl_v1"  # Mantenga este nombre para reanudar.
    for symbol in CFG.symbols:
        rl_run = f"{RL_RUN_TAG}_{RL_ALGO}_{symbol}" + ("_smoke" if SMOKE else "")
        rl_command = [sys.executable, "scripts/train_rl.py", *CONFIG_ARGS,
            "--algo", RL_ALGO, "--symbol", symbol, "--seeds", "0", "1", "2",
            "--train-end", "2024-01-01", "--eval-start", "2024-01-01",
            "--eval-end", "2025-11-01", "--device", RL_DEVICE,
            "--models-dir", str(CFG.models_dir), "--run", rl_run, "--resume"]
        if SMOKE:
            rl_command.append("--smoke")
        run_step(f"rl_{RL_ALGO}_{symbol}_{RL_DEVICE}", rl_command)
        rl_output = CFG.models_dir / "rl" / rl_run
        rl_summary = json.loads((rl_output / "eval_summary.json").read_text())
        display(pd.DataFrame({seed: result["metrics"]
                              for seed, result in rl_summary["seeds"].items()}).T)
        print("Comprar y mantener:", rl_summary["buy_and_hold"])
        print("Resultados en:", rl_output)

else:
    print("Etapa rl omitida (ver ETAPAS en la celda de parámetros).")


## 10. Reanudar, cambiar de GPU y reproducir
- Después de un corte, monte el mismo Drive, obtenga el mismo código y ejecute las celdas 0–3 y 7. No necesita repetir descarga, ML ni benchmark para reanudar.
- Conserve `RUN_TAG`, modelos, datos, épocas, batch, semilla e hiperparámetros. `last.pt` guarda la última época completa; `best.pt` guarda el mínimo `val_loss`. Se restaura RNG, optimizador, scheduler y normalización. Los archivos están en `folds/<símbolo>/fold_<k>/checkpoints/`.
- Cambie el tipo de acelerador en Colab y vuelva a conectar. El código detecta CUDA y elige bf16/fp16 de nuevo. Si el batch no cabe en la nueva GPU, regrese a una GPU compatible o inicie otro `RUN_TAG` con un batch menor; cambiar batch altera el experimento.
- Guarde el ZIP/commit y los archivos `environment_*.json`, `experiment.json`, `run_info.json` y métricas. Las semillas reducen variación; GPU, versiones y kernels diferentes pueden producir resultados numéricos distintos.
- En GRU/TCN v1 el tramo de calibración se usa para early stopping y Platt. En GRU/TCN v2 (`--separate-validation`) y en la CNN el early stopping usa una validación propia y la calibración solo ajusta Platt. Las métricas de calibración son diagnósticas. La comparación de estrategias usa exclusivamente predicciones del tramo de prueba. No use el holdout para elegir modelos.
- Si la sesión se corta durante la ronda v2, el último ZIP intermedio ya está en Drive (Colab) o en `/kaggle/working` (Kaggle).
- Kaggle (alternativa): lance la corrida completa con **Save Version → Save & Run All (Commit)**. Corre en segundo plano sin la pestaña abierta y guarda `/kaggle/working` como salida de esa versión. No se reanuda entre sesiones; revise la celda de resumen al final del log.
- Colab puede desconectar la sesión, cambiar el hardware o perder el disco temporal. No sirve para operar un bot 24/7. Este flujo produce artefactos de investigación en PAPER y no despliega órdenes reales.

## Análisis de gráficos (CNN)
Ventanas causales, validación temprana y calibración separadas. Cada imagen se pre-renderiza una sola vez por activo en la RAM de la VM (nunca en Drive) y la comparten todos los folds y épocas; la celda imprime cuántas imágenes, cuánto tardó y cuánta memoria ocupan (menos de 0.8 GB por activo con ventana 48). Es la misma imagen que antes se dibujaba en cada época; `--no-cache` vuelve a ese modo, mucho más lento. En SMOKE se borra antes la carpeta `chart_cnn_w48_smoke`.

In [ ]:
try:
    if ETAPAS["cnn"]:
        if not IN_CLOUD:
            raise RuntimeError("Ejecute el entrenamiento en Colab o Kaggle.")
        chart_out = CFG.models_dir / "dl" / ("chart_cnn_w48_smoke" if SMOKE else "chart_cnn_w48")
        if SMOKE and chart_out.exists():
            shutil.rmtree(chart_out)  # prueba mínima desechable: no se reanuda
        command = [sys.executable, "scripts/train_chart_cnn.py", *CONFIG_ARGS, "--window", "48", "--resume", "--out", str(chart_out)]
        if SMOKE:
            command.append("--smoke")
        run_step("chart_cnn", command)
        chart_summary = json.loads((chart_out / "backtest_summary.json").read_text())
        display(pd.DataFrame(chart_summary["strategies"]).T)
    else:
        print("Etapa cnn omitida (ver ETAPAS en la celda de parámetros).")

except Exception as error:  # noqa: BLE001 — una etapa caída no debe impedir consolidar ni exportar
    registrar_fallo("cnn", error)
if ETAPAS["cnn"]:
    export_results()  # el ZIP de resultados se actualiza tras cada etapa v2 (un corte no borra lo terminado)


## Evaluación final v2 (consolidación)
Re-evalúa juntos, con trato idéntico, todos los candidatos v2 (variantes ML, GRU/TCN v2 y CNN) sobre la ventana común: mismos costes, stops y referencias, DSR con todos los intentos acumulados, bootstrap pareado y puerta v2. Este es el reporte que cuenta.


In [ ]:
try:
    if ETAPAS["experimento_v2"] and not SMOKE:
        run_step("consolidacion_v2", [sys.executable, "scripts/consolidate_v2.py", *CONFIG_ARGS])
    else:
        print("Consolidación omitida (requiere el experimento v2 completo, sin SMOKE).")

except Exception as error:  # noqa: BLE001 — una etapa caída no debe impedir consolidar ni exportar
    registrar_fallo("consolidacion_v2", error)
if ETAPAS["experimento_v2"] and not SMOKE:
    export_results()  # el ZIP de resultados se actualiza tras cada etapa v2 (un corte no borra lo terminado)


## Resumen de la sesión
Lista cada etapa con su estado y duración y termina con «TODAS LAS ETAPAS TERMINARON BIEN» o con las etapas que fallaron y las últimas líneas de cada error. Se guarda en `reports/colab_sessions/resumen_<sesión>.json`, que viaja en el ZIP. Sirve sobre todo en Kaggle con *Save & Run All*, donde nadie ve los errores intermedios.

In [ ]:
elapsed = time.perf_counter() - SESSION_STARTED
entorno = "Colab" if IN_COLAB else "Kaggle" if IN_KAGGLE else "local"
print(f"Sesión {SESSION_ID} | {entorno} | SMOKE={SMOKE} | {elapsed / 3600:.2f} h desde la celda 0")
for step in STEPS:
    print(f"  {step['step']:<30} {step['status']:<16} {step['seconds'] / 60:7.1f} min")
resumen = {"session": SESSION_ID, "entorno": entorno, "smoke": SMOKE, "etapas": ETAPAS,
           "elapsed_seconds": elapsed, "units_per_hour": UNITS_PER_HOUR,
           "estimated_units": elapsed / 3600 * float(UNITS_PER_HOUR) if UNITS_PER_HOUR is not None else None,
           "steps": STEPS, "fallos": FALLOS, "todo_bien": not FALLOS}
resumen_path = CFG.reports_dir / "colab_sessions" / f"resumen_{SESSION_ID}.json"
resumen_path.write_text(json.dumps(resumen, indent=2, ensure_ascii=False), encoding="utf-8")
print("Resumen guardado en", resumen_path)
if FALLOS:
    print(f"\nFALLARON {len(FALLOS)} ETAPA(S):")
    for fallo in FALLOS:
        print(f"- {fallo['etapa']}: {fallo['tipo']}: {fallo['mensaje'].splitlines()[0] if fallo['mensaje'] else ''}")
        for line in fallo["ultimas_lineas"][-15:]:
            print("      " + line)
else:
    print("\nTODAS LAS ETAPAS TERMINARON BIEN")


## Exportar resultados al PC
Empaqueta reportes, métricas, predicciones, benchmarks y el registro de modelos en un solo ZIP (sin checkpoints `.pt` ni modelos RL pesados). Es el mismo archivo que se fue actualizando tras cada etapa v2. En Colab queda en Drive (`DRIVE_DIR`) y se descarga en tu navegador; en Kaggle queda en `/kaggle/working`, visible en la pestaña *Output*. Pásale ese archivo a Claude para analizarlo.

In [ ]:
export_results(final=True)
if IN_COLAB:
    from google.colab import files
    files.download(str(EXPORT_ZIP))
elif IN_KAGGLE:
    print("Kaggle: descárgalo desde el panel derecho (Output > /kaggle/working) o, si corriste con Save Version, "
          "desde la pestaña Output de esa versión del notebook.")
